# BDA 제2회 학습자 수료 예측 — 최종 제출 노트북

| 항목 | 내용 |
|------|------|
| 대회 | BDA × DACON 학습자 수료 예측 AI 경진대회 |
| 최종 성적 | Private 상위 4% (733명 중 29위), F1 Score 0.40921 |
| 최종 제출 파일 | `catboost_tuning_v3` |
| 모델 | CatBoost + Optuna (500 trials) + RepeatedStratifiedKFold(5×3) |
| 피처셋 | V4 파생변수 (수료 시그널 기반, Zero-Importance 5개 제거) |

## 실행 순서
1. 환경 설정 & 라이브러리 로드
2. 데이터 로드 & 전처리 (`Pipeline`)
3. Optuna 하이퍼파라미터 튜닝 (`Tuner`)
4. 최종 예측 & 제출 파일 저장 (`predict_params`)

In [ ]:
import os
import warnings
from pathlib import Path
import pandas as pd
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'jupyter' else Path.cwd()
os.chdir(PROJECT_ROOT)
print(f'Project Root: {PROJECT_ROOT}')

In [ ]:
from src.preprocess.pipeline import Pipeline
from src.models import Tuner, predict_params

In [ ]:
# ── 데이터 로드 & 전처리 ─────────────────────────────────────

train_raw = pd.read_csv('dataset/train.csv')
test_raw  = pd.read_csv('dataset/test.csv')
print(f'train: {train_raw.shape}, test: {test_raw.shape}')

pipe = Pipeline(config={'skip_steps': ['encode_text_embeddings']})
tr, te, tid = pipe.run(train_raw, test_raw)

# feature importance=0 컬럼 및 nationality 제거 (v4 기준)
DROP_COLS = ['class4', 'dcert_컴퓨터활용능력', 'odc_AI모델', 'odc_Hadoop', 'nationality']

X  = tr.drop(columns=['completed'] + [c for c in DROP_COLS if c in tr.columns])
y  = tr['completed']
te = te.drop(columns=[c for c in DROP_COLS if c in te.columns])

print(f'X: {X.shape}, y: {y.shape}, pos_rate: {y.mean():.3f}')

In [ ]:
# ── Optuna 튜닝 ───────────────────────────────────────────────

N_TRIALS = 500

tuner = Tuner('catboost', X, y, pipeline=pipe, n_trials=N_TRIALS)
study = tuner.run()

In [ ]:
# ── 최종 예측 & 제출 파일 저장 ────────────────────────────────

oof, test_probs, best_t = predict_params(
    'catboost', tuner.best_params,
    X, y, te, tid,
    pipeline=pipe,
    version='final'
)